# Classificação das colunas — Numérica vs. Categórica

Insumo para a Seção 4.2.1 da documentação (Exploração de dados).

Fonte: `data/dataset_long.csv`, gerado por [`00_reestruturacao_planilha.ipynb`](00_reestruturacao_planilha.ipynb) — já está no formato "tidy" (uma linha por combinação Run x TC), então este notebook **não altera dado nenhum**, só lê e classifica. Por isso fica separado de `01_exploracao_dados.ipynb`/`02_pre_processamento.ipynb`, que são os notebooks reservados para quem manipula o dado real.

Metodologia: `df.dtypes` como ponto de partida, seguido de revisão manual coluna a coluna para pegar casos ambíguos (ex.: coluna que parece numérica mas é identificador, ou tem valores textuais misturados).

## 1. Carregar o dataset tidy

In [1]:
import pandas as pd

df = pd.read_csv('../data/dataset_long.csv')
df.shape

(4065, 14)

## 2. Baseline com `df.dtypes`

Ponto de partida obrigatório pela task — mas o pandas infere tipo pela sintaxe do valor, não pelo significado da coluna, então precisa de revisão manual em seguida.

In [2]:
df.dtypes

run_id                       str
tc_number                  int64
tc_name                      str
key_area                     str
active_limit                 str
temp                     float64
reason                       str
blower_direction             str
housing                      str
board_position               str
run_case                     str
losses                       str
deflector_hole               str
material_coil_support        str
dtype: object

## 3. Revisão manual das colunas que parecem numéricas

`df.dtypes` mostra tudo como `str`/`object` porque o CSV tem sujeira de formatação (identificadores misturados com número, texto onde deveria ter número, etc.). Antes de classificar, checamos com `pd.to_numeric(..., errors='coerce')` quais colunas *realmente* convertem para número de forma limpa, e o que sobra quando não convertem — isso é o que decide se a coluna é Numérica de fato ou só parece.

In [3]:
candidatas_numericas = ['tc_number', 'active_limit', 'temp']

for col in candidatas_numericas:
    coerced = pd.to_numeric(df[col], errors='coerce')
    nao_numericos = df[col][coerced.isna() & df[col].notna()].unique()
    print(f"{col}: {len(nao_numericos)} valores nao numericos -> {list(nao_numericos)[:10]}")

tc_number: 0 valores nao numericos -> []
active_limit: 1 valores nao numericos -> ['Monitor (150 C)']
temp: 0 valores nao numericos -> []


**Leitura dos 3 casos:**

- `tc_number`: já converte 100% para número (o `#REF!` que existia numa versão anterior do dataset foi corrigido na reestruturação). Mesmo assim, continua sendo um **identificador** do sensor, não uma quantidade — não faz sentido calcular média/desvio de `tc_number`. → **Categórica**.
- `active_limit`: ainda resta 1 valor não numérico (`"Monitor (150 C)"`) — sensor sem limite de segurança definido (só monitorado, sem threshold). O núcleo da coluna é uma temperatura limite em °C. → **Numérica**, mas esse valor residual precisa virar `NaN` (ou um flag `is_monitor` à parte) no pré-processamento (Seção 4.2.2) antes de qualquer cálculo estatístico.
- `temp`: já converte 100% para número (as strings formatadas como porcentagem que existiam numa versão anterior do dataset foram corrigidas na reestruturação). → **Numérica**, sem ressalvas.

As demais colunas (`run_id`, `tc_name`, `key_area`, `reason`, `blower_direction`, `housing`, `board_position`, `run_case`, `losses`, `deflector_hole`, `material_coil_support`) são texto/rótulo por natureza — sem ambiguidade, todas **Categóricas**.

## 4. Tabela final de classificação

In [4]:
# colunas que sao de fato Numericas depois da revisao manual (secao 3);
# tudo que nao estiver aqui e Categorica, mesmo que df.dtypes tenha mostrado
# algo parecido com numero (caso do tc_number)
NUMERICAS = {'active_limit', 'temp'}

rows = []
for col in df.columns:
    tipo = 'Numerica' if col in NUMERICAS else 'Categorica'
    rows.append({
        'coluna': col,
        'tipo': tipo,
        'n_unicos': int(df[col].nunique()),
        'ausentes': int(df[col].isna().sum()),
    })

classificacao = pd.DataFrame(rows)
classificacao

,coluna,tipo,n_unicos,ausentes
0,run_id,Categorica,56,0
1,tc_number,Categorica,77,0
2,tc_name,Categorica,77,0
3,key_area,Categorica,7,0
4,active_limit,Numerica,18,0
5,temp,Numerica,3213,0
6,reason,Categorica,34,134
7,blower_direction,Categorica,4,0
8,housing,Categorica,3,0
9,board_position,Categorica,3,0


In [5]:
classificacao.to_csv('../data/classificacao_colunas.csv', index=False)
print('salvo em data/classificacao_colunas.csv')

salvo em data/classificacao_colunas.csv


## 5. Markdown pronto para a Seção 4.2.1

In [6]:
def to_markdown(d, cols):
    header = '| ' + ' | '.join(cols) + ' |'
    sep = '|' + '|'.join(['---'] * len(cols)) + '|'
    linhas = [header, sep]
    for _, row in d.iterrows():
        linhas.append('| ' + ' | '.join(str(row[c]) for c in cols) + ' |')
    return '\n'.join(linhas)

print(to_markdown(classificacao, ['coluna', 'tipo']))

| coluna | tipo |
|---|---|
| run_id | Categorica |
| tc_number | Categorica |
| tc_name | Categorica |
| key_area | Categorica |
| active_limit | Numerica |
| temp | Numerica |
| reason | Categorica |
| blower_direction | Categorica |
| housing | Categorica |
| board_position | Categorica |
| run_case | Categorica |
| losses | Categorica |
| deflector_hole | Categorica |
| material_coil_support | Categorica |
